# HW1: Build & Prompt a Recipe Agent

In this homework you'll build a recipe suggestion bot with LangChain + Tavily, traced to LangSmith.

**What you'll learn:**
- How to create a tool-calling agent with `create_agent`
- How traces automatically appear in LangSmith
- How to bulk-test an agent with diverse queries

## Setup

Make sure your `.env` file has:
```
OPENAI_API_KEY=...
LANGSMITH_API_KEY=...
TAVILY_API_KEY=...
LANGSMITH_TRACING=true
```

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 1. Define the Agent

We use `create_agent` from LangChain with a Tavily web search tool.
The agent definition is also extracted to `agent.py` so later homeworks can import it.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from tavily import TavilyClient

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> dict:
    """Search the web for recipes, ingredients, and cooking information."""
    return tavily_client.search(query)


system_prompt = """\
You are a helpful recipe suggestion assistant. The user will describe what \
ingredients they have, dietary preferences, cuisine interests, or ask general \
cooking questions.

Using the web search tool, find relevant recipes and provide clear, actionable \
suggestions. When recommending a recipe, include:
- Recipe name
- Key ingredients (noting any the user already has)
- Brief preparation overview
- Any relevant dietary notes (allergens, substitutions)

Be friendly, concise, and practical.\
"""

agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search],
    system_prompt=system_prompt,
)

## 2. Run a Sample Query

Every invocation is automatically traced to LangSmith. After running this cell, open [LangSmith](https://smith.langchain.com) and look for the trace in your default project.

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="I have chicken thighs, rice, and bell peppers. What can I make?")]}
)

print(response["messages"][-1].content)

## 3. Bulk-Test with Diverse Queries

Let's run a set of diverse queries to exercise the agent across different scenarios:
dietary restrictions, cuisine types, ingredient constraints, and edge cases.

In [ ]:
test_queries = [
    "I have leftover salmon and some cream cheese. Any ideas?",
    "What's a good vegan dinner I can make in under 30 minutes?",
    "I'm gluten-free and craving Italian food. Suggestions?",
    "Give me a keto-friendly lunch recipe with ground beef.",
    "I only have eggs, butter, and flour. What can I bake?",
    "Suggest a Thai curry recipe for beginners.",
    "What's a healthy meal-prep recipe I can make on Sunday for the whole week?",
    "I have a bunch of zucchini from my garden. What should I do with it?",
    "Recommend a kid-friendly dinner that's also nutritious.",
    "I want to make a fancy dessert for a dinner party. What do you suggest?",
    "What's a traditional Mexican breakfast recipe?",
    "I'm dairy-free and want to make mac and cheese. How?",
    "Suggest a high-protein vegetarian recipe.",
    "I have canned chickpeas, spinach, and garlic. What can I make?",
    "What's an easy recipe for homemade pasta sauce?",
]

In [ ]:
for i, query in enumerate(test_queries):
    print(f"\n{'='*60}")
    print(f"Query {i+1}: {query}")
    print('='*60)

    response = agent.invoke(
        {"messages": [HumanMessage(content=query)]}
    )

    # Print a truncated version of the response
    answer = response["messages"][-1].content
    print(answer[:500] + ("..." if len(answer) > 500 else ""))

## 4. View Traces in LangSmith

Head over to [smith.langchain.com](https://smith.langchain.com) and open your project.
You should see all 16 traces (1 sample + 15 bulk).

For each trace you can inspect:
- The full message history
- Tool calls made (web search queries)
- Latency and token usage
- The raw inputs and outputs at each step